# ADBA QLoRA — RTX 5060 Ti (Windows 10 / Python 3.12)

Fine-tune `Qwen2.5-Coder-7B-Instruct` bằng QLoRA.

**Yêu cầu:** đặt notebook trong folder có cấu trúc:
```
project/
  data/
    train.jsonl
    valid.jsonl
    test.jsonl
  adapters/          ← tự tạo
  train_qlora.ipynb  ← notebook này
```

In [1]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 0 — PHẢI CHẠY ĐẦU TIÊN TRƯỚC MỌI IMPORT      ║
# ║  Fix Windows encoding (cp1252 → utf-8)               ║
# ╚══════════════════════════════════════════════════════╝
import os, sys

os.environ['PYTHONUTF8']       = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

print('stdout encoding :', sys.stdout.encoding)
print('Python version  :', sys.version)
print('Platform        :', sys.platform)


stdout encoding : UTF-8
Python version  : 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Platform        : win32


In [2]:
import subprocess, sys

# Gỡ torch cũ
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'torch', 'torchvision', 'torchaudio', '-y'])

# Cài PyTorch nightly — hỗ trợ CUDA 13.x / RTX 5060 Ti
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    '--pre', 'torch', 'torchvision', 'torchaudio',
    '--index-url', 'https://download.pytorch.org/whl/nightly/cu128',
    '--upgrade', '-q'
], check=True)

print('Done! Restart kernel sau do.')

Done! Restart kernel sau do.


In [2]:
%pip install datasets

   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ---------------------------------------- 0.0/529.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/529.0 kB ? eta -:--:--
   ------------------- -------------------- 262.1/529.0 kB ? eta -:--:--
   -------------------------------------- 529.0/529.0 kB 502.5 kB/s eta 0:00:00
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   --------------- ------------------------ 262.1/663.6 kB ? eta -:--:--
   --------------- ------------------------ 262.1/663.6 kB ? eta -:--:--
   --------------- ------------------------ 262.1/663.6 kB ?

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install -U peft transformers trl accelerate bitsandbytes datasets

  Using cached peft-0.19.1-py3-none-any.whl.metadata (15 kB)
   ---------------------------------------- 0.0/680.7 kB ? eta -:--:--
   --------------- ------------------------ 262.1/680.7 kB ? eta -:--:--
   ---------------------------------------- 680.7/680.7 kB 3.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.6 MB 3.4 MB/s eta 0:00:04
   -- ------------------------------------- 0.8/10.6 MB 2.4 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/10.6 MB 2.1 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/10.6 MB 2.0 MB/s eta 0:00:05
   ------- -------------------------------- 2.1/10.6 MB 2.0 MB/s eta 0:00:05
   --------- ------------------------------ 2.6/10.6 MB 2.0 MB/s eta 0:00:05
   ---------- ----------------------------- 2.9/10.6 MB 2.0 MB/s eta 0:00:04
   ----------- ---------------------------- 3.1/10.6 MB 1.9 MB/s eta 0:00:04
   ------------ -------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ╔═══════════════════════════════════╗
# ║  CELL 2 — Imports & env setup    ║
# ╚═══════════════════════════════════╝
import os, sys, json, gc
from pathlib import Path

# ── Windows UTF-8 fix ──────────────────────────────────────
os.environ['PYTHONUTF8']       = '1'
os.environ['PYTHONIOENCODING'] = 'utf-8'
os.environ['TOKENIZERS_PARALLELISM']            = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_DATASETS_DISABLE_PROGRESS_BAR'] = '0'

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')
if hasattr(sys.stderr, 'reconfigure'):
    sys.stderr.reconfigure(encoding='utf-8')

# ── PATCH pathlib.Path.read_text để luôn dùng utf-8 ────────
# TRL đọc file .jinja bằng Path.read_text() không truyền encoding
# → Windows dùng cp1252 → crash. Patch này fix triệt để.
_original_read_text = Path.read_text
def _utf8_read_text(self, encoding=None, errors=None):
    return _original_read_text(self, encoding=encoding or 'utf-8', errors=errors)
Path.read_text = _utf8_read_text
print('pathlib patch : OK')

# ── Imports ─────────────────────────────────────────────────
import torch
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from trl import SFTConfig, SFTTrainer   # ← sẽ không còn crash

# ── DataCollator tự implement ────────────────────────────────
class DataCollatorForCompletionOnlyLM:
    """Mask labels phần prompt, chỉ train trên assistant reply."""
    def __init__(self, response_template: str, tokenizer):
        self.tpl_ids   = tokenizer.encode(response_template, add_special_tokens=False)
        self.tokenizer = tokenizer

    def __call__(self, features):
        from torch.nn.utils.rnn import pad_sequence
        input_ids_list, labels_list = [], []
        for f in features:
            ids = f['input_ids']
            if not isinstance(ids, list):
                ids = ids.tolist()
            labels   = list(ids)
            tpl      = self.tpl_ids
            last_pos = -1
            for i in range(len(ids) - len(tpl) + 1):
                if ids[i : i + len(tpl)] == tpl:
                    last_pos = i
            if last_pos >= 0:
                for j in range(last_pos + len(tpl)):
                    labels[j] = -100
            else:
                labels = [-100] * len(labels)
            input_ids_list.append(torch.tensor(ids))
            labels_list.append(torch.tensor(labels))
        input_ids      = pad_sequence(input_ids_list, batch_first=True,
                                      padding_value=self.tokenizer.pad_token_id)
        labels         = pad_sequence(labels_list, batch_first=True, padding_value=-100)
        attention_mask = (input_ids != self.tokenizer.pad_token_id).long()
        return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

# ── Kiểm tra GPU ─────────────────────────────────────────────
print('torch        :', torch.__version__)
print('cuda         :', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('gpu          :', props.name)
    print('vram         :', round(props.total_memory/1e9, 1), 'GB')
    print('bf16 support :', torch.cuda.is_bf16_supported())
else:
    raise RuntimeError('CUDA not available!')

pathlib patch : OK


c:\Users\ezycloudx-admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch        : 2.12.0.dev20260408+cu128
cuda         : True
gpu          : NVIDIA GeForce RTX 5090
vram         : 34.2 GB
bf16 support : True


In [2]:
# ╔══════════════════════════╗
# ║  CELL 3 — Paths & config ║
# ╚══════════════════════════╝
CANDIDATES = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_DIR = None
for c in CANDIDATES:
    if (c / 'data' / 'train.jsonl').exists():
        PROJECT_DIR = c
        break
if PROJECT_DIR is None:
    raise FileNotFoundError('Khong tim thay data/train.jsonl. Mo notebook trong folder project.')

TRAIN_PATH = PROJECT_DIR / 'data' / 'train.jsonl'
VALID_PATH = PROJECT_DIR / 'data' / 'valid.jsonl'
TEST_PATH  = PROJECT_DIR / 'data' / 'test.jsonl'
OUT_DIR    = PROJECT_DIR / 'adapters' / 'qwen25-coder-7b-adba-qlora'
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL     = 'Qwen/Qwen2.5-Coder-7B-Instruct'
MAX_SEQ_LENGTH = 1024         # 5060 Ti 16GB: safe với batch=2
BF16           = torch.cuda.is_bf16_supported()
TORCH_DTYPE    = torch.bfloat16 if BF16 else torch.float16
OPTIMIZER      = 'paged_adamw_8bit'

print('PROJECT_DIR :', PROJECT_DIR)
print('OUT_DIR     :', OUT_DIR)
print('dtype       :', TORCH_DTYPE)
print('bf16        :', BF16)


PROJECT_DIR : C:\Users\ezycloudx-admin\Downloads\train
OUT_DIR     : C:\Users\ezycloudx-admin\Downloads\train\adapters\qwen25-coder-7b-adba-qlora
dtype       : torch.bfloat16
bf16        : True


In [3]:
# ╔═══════════════════╗
# ║  CELL 4 — Data   ║
# ╚═══════════════════╝
def load_jsonl(path: Path):
    rows = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
valid_rows = load_jsonl(VALID_PATH)
test_rows  = load_jsonl(TEST_PATH)

print('train:', len(train_rows))
print('valid:', len(valid_rows))
print('test :', len(test_rows))
print('roles:', [m['role'] for m in train_rows[0]['messages']])


train: 787
valid: 99
test : 98
roles: ['system', 'user', 'assistant']


In [4]:
# ╔══════════════════════════╗
# ║  CELL 5 — Tokenizer & Model ║
# ╚══════════════════════════╝
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=TORCH_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=TORCH_DTYPE,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model.enable_input_require_grads()

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# VRAM check
gc.collect()
torch.cuda.empty_cache()
for i in range(torch.cuda.device_count()):
    alloc = torch.cuda.memory_allocated(i)/1e9
    total = torch.cuda.get_device_properties(i).total_memory/1e9
    print(f'GPU {i}: {alloc:.1f}/{total:.1f} GB used')


W0516 15:12:29.428000 8048 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]c:\Users\ezycloudx-admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ezycloudx-admin\.cache\huggingface\hub\models--Qwen--Qwen2.5-Coder-7B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see th

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273
GPU 0: 7.9/34.2 GB used


In [5]:
# ╔══════════════════════════╗
# ║  CELL 6 — Dataset format ║
# ╚══════════════════════════╝
def format_example(example):
    text = tokenizer.apply_chat_template(
        example['messages'], tokenize=False, add_generation_prompt=False
    )
    return {'text': text}

train_ds = Dataset.from_list(train_rows).map(format_example, remove_columns=['messages'])
valid_ds = Dataset.from_list(valid_rows).map(format_example, remove_columns=['messages'])
test_ds  = Dataset.from_list(test_rows).map(format_example,  remove_columns=['messages'])

data_collator = DataCollatorForCompletionOnlyLM(
    response_template='<|im_start|>assistant\n',
    tokenizer=tokenizer,
)

print(train_ds)
print('Preview:')
print(train_ds[0]['text'][:600])


Map: 100%|██████████| 98/98 [00:00<00:00, 12908.39 examples/s]

Dataset({
    features: ['text'],
    num_rows: 787
})
Preview:
<|im_start|>system
## ROLE
You are the Supervisor Agent for ADBA (Autonomous Data & Business Intelligence Agent).
Your only job is to read a user query and produce a valid ExecutionPlan JSON that routes
work to the correct specialist agents.

## AVAILABLE AGENTS
- sql     → retrieves data from PostgreSQL (skill: text-to-sql)
- python  → transforms / analyzes a DataFrame (skill: data-analysis)
- viz     → generates a chart from a DataFrame (skill: visualization)
- insight → produces a structured business insight from all results (skill: insight-generation)

## ROUTING RULES
1. Always start with


In [6]:
# ╔══════════════════════╗
# ║  CELL 7 — SFTTrainer ║
# ╚══════════════════════╝
training_args = SFTConfig(
    output_dir=str(OUT_DIR),
    dataset_text_field='text',
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 2x8 = 16
    learning_rate=2e-4,
    num_train_epochs=3,
    warmup_steps=20,
    lr_scheduler_type='cosine',
    optim=OPTIMIZER,
    logging_steps=5,
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=25,
    save_steps=25,
    save_total_limit=3,              # giữ 3 checkpoint để resume an toàn
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    bf16=BF16,
    fp16=not BF16,
    gradient_checkpointing=True,
    disable_tqdm=False,
    packing=False,
    report_to='none',
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)
print(trainer)


Tokenizing eval dataset: 100%|██████████| 99/99 [00:00<00:00, 375.40 examples/s]

In [7]:
# ╔══════════════════════════════════════════════════════╗
# ║  CELL 8 — Train (tự động resume nếu có checkpoint)  ║
# ╚══════════════════════════════════════════════════════╝
gc.collect()
torch.cuda.empty_cache()

# Tìm checkpoint mới nhất để resume
ckpt_dirs = sorted(
    [d for d in OUT_DIR.glob('checkpoint-*') if d.is_dir()],
    key=lambda p: int(p.name.split('-')[1])
)

if ckpt_dirs:
    latest_ckpt = str(ckpt_dirs[-1])
    print(f'Resuming from: {latest_ckpt}')
    train_result = trainer.train(resume_from_checkpoint=latest_ckpt)
else:
    print('Khong co checkpoint, train tu dau...')
    train_result = trainer.train()

train_metrics = train_result.metrics
print(train_metrics)


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Khong co checkpoint, train tu dau...


Step,Training Loss,Validation Loss
25,0.201749,0.194935
50,0.146930,0.161612


AcceleratorError: CUDA error: unknown error
Search for `cudaErrorUnknown' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# ╔════════════════════╗
# ║  CELL 9 — Evaluate ║
# ╚════════════════════╝
eval_metrics = trainer.evaluate(valid_ds)
print(eval_metrics)


In [ ]:
# ╔═══════════════════════════════╗
# ║  CELL 10 — Save adapter + zip ║
# ╚═══════════════════════════════╝
import shutil

trainer.save_model(str(OUT_DIR))
tokenizer.save_pretrained(str(OUT_DIR))

metrics_path = OUT_DIR / 'metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump({'train': train_metrics, 'valid': eval_metrics}, f, ensure_ascii=False, indent=2)

# Zip để copy về máy local
zip_path = shutil.make_archive(str(OUT_DIR), 'zip', root_dir=str(OUT_DIR))
print('Adapter saved :', OUT_DIR)
print('ZIP created   :', zip_path)
print('Files:')
for p in sorted(OUT_DIR.glob('*')):
    print(' -', p.name)


In [ ]:
# ╔═══════════════════════════════════════════════╗
# ║  CELL 11 — Smoke test trên holdout (optional) ║
# ╚═══════════════════════════════════════════════╝
model.eval()

for idx in range(min(3, len(test_rows))):
    prompt_messages = test_rows[idx]['messages'][:-1]
    reference       = test_rows[idx]['messages'][-1]['content']

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt_text, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True
    )
    print('=' * 80)
    print(f'HOLDOUT {idx}')
    print('PROMPT  :', prompt_messages[-1]['content'][:300])
    print('PREDICT :', generated[:800])
    print('REF     :', reference[:800])


In [ ]:
# ╔═════════════════════════════════════════════════════════╗
# ║  CELL 12 — Backup checkpoint (chạy trước khi hết giờ)  ║
# ╚═════════════════════════════════════════════════════════╝
import shutil

ckpt_dirs = sorted(
    [d for d in OUT_DIR.glob('checkpoint-*') if d.is_dir()],
    key=lambda p: int(p.name.split('-')[1])
)

if ckpt_dirs:
    latest = ckpt_dirs[-1]
    zip_path = shutil.make_archive(
        str(OUT_DIR.parent / f'BACKUP_{latest.name}'), 'zip',
        root_dir=str(latest)
    )
    size_mb = Path(zip_path).stat().st_size / 1e6
    print(f'Checkpoint backup: {zip_path}')
    print(f'Size: {size_mb:.0f} MB')
    print('Hay tai file nay ve may local ngay!')
else:
    print('Khong co checkpoint nao de backup.')
